# Этап 1. Ответы модели и её скрытые состояния

Ноутбук для Google Colab. **Среда выполнения → Сменить среду → GPU T4.**

Что делает: прогоняет языковую модель по 2327 вопросам RuBQ на русском и английском, сохраняет ответы,
признаки неуверенности и скрытые состояния всех слоёв. Результаты пишутся на Google Диск,
поэтому если Colab отключится, просто запустите ячейку прогона ещё раз — она продолжит с места остановки.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/NilsonML/hallucination-probe.git"  # свой репозиторий
OUT = "/content/drive/MyDrive/hallucination-probe/outputs"         # результаты на Диске

In [ ]:
!git clone -q $REPO_URL /content/hallucination-probe || (cd /content/hallucination-probe && git pull -q)
%cd /content/hallucination-probe
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Быстрая проверка на 32 вопросах (1–2 минуты)

In [ ]:
!PYTHONPATH=src python -m hprobe.generate --model Qwen/Qwen2.5-0.5B-Instruct --lang ru --limit 32 --out /content/smoke
!PYTHONPATH=src python -m hprobe.label --run /content/smoke
import pandas as pd
pd.read_csv('/content/smoke/labels.csv')[['question','answer','gold','correct','consistency']].head(15)

## 2. Основные прогоны

Порядок важен: сначала главная модель (1.5B) на обоих языках, затем 0.5B и 3B для эксперимента
с размером модели. Ориентировочно 15–30 минут на прогон 1.5B на T4.
Если GPU не хватает памяти, уменьшите `--batch-size`.

In [ ]:
RUNS = [
    ("Qwen/Qwen2.5-1.5B-Instruct", "qwen2.5-1.5b", 16),
    ("Qwen/Qwen2.5-0.5B-Instruct", "qwen2.5-0.5b", 32),
    ("Qwen/Qwen2.5-3B-Instruct",   "qwen2.5-3b",   8),
]
for model, short, bs in RUNS:
    for lang in ["ru", "en"]:
        out = f"{OUT}/{short}/{lang}"
        !PYTHONPATH=src python -m hprobe.generate --model {model} --lang {lang} --out {out} --batch-size {bs}
        !PYTHONPATH=src python -m hprobe.label --run {out}

## 3. Ручная проверка разметки

В папке каждого прогона появился `manual_check.csv` (100 случайных ответов). Для главного прогона
`qwen2.5-1.5b/ru` откройте файл (например, в Google Таблицах), в столбце `human` поставьте 1, если ответ
модели верный, и 0, если неверный. Сохраните как CSV на то же место и запустите ячейку ниже:
получится точность автоматической разметки — это число пойдёт в тезисы.

In [ ]:
!PYTHONPATH=src python -m hprobe.label --run {OUT}/qwen2.5-1.5b/ru --check

## 4. Первые результаты: пробы по слоям

Для каждого прогона обучаются пробы по всем слоям и сравниваются с базовыми методами.
GPU не нужен. График `probe/layers.png` и таблица `probe/summary.csv` сохраняются в папке прогона.

In [ ]:
import glob
from IPython.display import Image, display
for run in sorted(glob.glob(f"{OUT}/*/*")):
    if not glob.os.path.exists(f"{run}/hidden.npy"):
        continue
    !PYTHONPATH=src python -m hprobe.probes --run {run}
    display(Image(f"{run}/probe/layers.png", width=600))